# Task 7 — User Preferences and Interaction Data

**Objective:** learn how user actions can be stored and build a small practice dataset for collaborative filtering.

> **Important:** the supplied TED Talks CSV has no real user ratings, likes, or watch history. The interaction rows created here are **simulated learning data**, not real TED user behaviour and not evidence of actual user preferences. The TED CSV `ratings` field contains aggregate reaction types and counts; it does not identify individual users and cannot be used as a user-item rating history.

In [7]:
import numpy as np
import pandas as pd

In [8]:
from pathlib import Path
import pandas as pd


def find_project_root():
    """Find the project folder from the current Jupyter working directory."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "ted_talks_merged.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/raw/ted_talks_merged.csv. "
        "Open Jupyter from the TED_Talks_Recommendation_Ajay project folder."
    )


PROJECT_DIR = find_project_root()
RAW_DATA_PATH = PROJECT_DIR / "data" / "raw" / "ted_talks_merged.csv"
PROCESSED_DATA_PATH = PROJECT_DIR / "data" / "processed" / "ted_talks_cleaned.csv"
print("Project folder:", PROJECT_DIR)

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError("Run Task 3 first.")
talks = pd.read_csv(PROCESSED_DATA_PATH)
needed = ["talk_id", "title", "combined_text"]
missing = [column for column in needed if column not in talks.columns]
if missing:
    raise ValueError(f"Cleaned dataset is missing columns: {missing}")
talks["combined_text"] = talks["combined_text"].fillna("").astype(str)
print("Available talks:", len(talks))
display(talks[["talk_id", "title"]].head())

Project folder: C:\Users\Ajay\Desktop\TED_Talks_Recommendation_Ajay
Available talks: 2550


,talk_id,title
0,Ken Robinson: Do schools kill creativity?,Do schools kill creativity?
1,Al Gore: Averting the climate crisis,Averting the climate crisis
2,David Pogue: Simplicity sells,Simplicity sells
3,Majora Carter: Greening the ghetto,Greening the ghetto
4,Hans Rosling: The best stats you've ever seen,The best stats you've ever seen


## 1. Understand interaction data

- `user_id`: a user in the practice system.
- `talk_id`: a TED Talk identifier that connects the interaction to the talks catalogue.
- `rating`: an explicit score from 1 to 5.
- `liked`: a simple like indicator derived from the rating in this exercise.
- `watched`: indicates that an interaction/watch is recorded.

An interaction matrix has users as rows, talks as columns, and ratings as values. A blank cell means no rating/interactions are recorded for that user-talk pair; it does not mean a rating of zero.

## 2. Create a reproducible simulated dataset

To make the exercise reproducible, we use a fixed random seed. Users are assigned to simple interest groups and are given simulated ratings for sampled talks. The groups are only a teaching device. They are not inferred real users.

In [9]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Use a modest catalogue subset so the user-item matrix is easy to inspect.
number_of_talks = min(100, len(talks))
catalog = talks.sample(n=number_of_talks, random_state=RANDOM_SEED).reset_index(drop=True)

interest_groups = {
    "technology": ["technology", "computer", "internet", "software", "design", "data", "robot"],
    "science_health": ["science", "health", "brain", "medicine", "climate", "energy", "research"],
    "education_society": ["education", "school", "culture", "society", "children", "learning", "community"],
}

# Estimate a simple text-theme score for each selected talk. This is only used to create simulated ratings.
text_values = catalog["combined_text"].str.lower()
for group, keywords in interest_groups.items():
    catalog[f"_{group}_score"] = text_values.apply(lambda text: sum(text.count(word) for word in keywords))

group_names = list(interest_groups)
user_rows = []
number_of_users = 18
interactions_per_user = min(45, number_of_talks)

for user_number in range(1, number_of_users + 1):
    group = group_names[(user_number - 1) % len(group_names)]
    sampled_positions = rng.choice(number_of_talks, size=interactions_per_user, replace=False)
    for position in sampled_positions:
        talk = catalog.iloc[int(position)]
        theme_scores = [talk[f"_{candidate}_score"] for candidate in group_names]
        talk_group = group_names[int(np.argmax(theme_scores))] if max(theme_scores) > 0 else "general"
        interest_bonus = 0.8 if talk_group == group else 0.0
        noise = rng.normal(0, 0.65)
        rating = int(np.clip(np.rint(3.0 + interest_bonus + noise), 1, 5))
        user_rows.append({
            "user_id": f"user_{user_number:02d}",
            "talk_id": talk["talk_id"],
            "rating": rating,
            "liked": bool(rating >= 4),
            "watched": True,
            "simulated_interest_group": group,
        })

interactions = pd.DataFrame(user_rows)
# One interaction per user-talk pair is expected by the pivot operation.
interactions = interactions.drop_duplicates(subset=["user_id", "talk_id"], keep="last")
print("Interaction rows:", len(interactions))
print("Unique users:", interactions["user_id"].nunique())
print("Unique talks with interactions:", interactions["talk_id"].nunique())
print("This dataset is simulated:", True)
display(interactions.head(10))

Interaction rows: 810
Unique users: 18
Unique talks with interactions: 100
This dataset is simulated: True


,user_id,talk_id,rating,liked,watched,simulated_interest_group
0,user_01,Meaghan Ramsey: Why thinking you're ugly is ba...,3,False,True,technology
1,user_01,"Silk Road Ensemble: ""Turceasca""",3,False,True,technology
2,user_01,Steve Ramirez and Xu Liu: A mouse. A laser bea...,3,False,True,technology
3,user_01,Morgana Bailey: The danger of hiding who you are,4,True,True,technology
4,user_01,Erik Hersman: Reporting crisis via texting,4,True,True,technology
5,user_01,Kevin Bales: How to combat modern slavery,3,False,True,technology
6,user_01,Rives: The Museum of Four in the Morning,3,False,True,technology
7,user_01,Mikko Hypponen: How the NSA betrayed the world...,4,True,True,technology
8,user_01,"Leah Buechley: How to ""sketch"" with electronics",4,True,True,technology
9,user_01,Stefan Sagmeister: The power of time off,3,False,True,technology


## 3. Save the interaction dataset

In [10]:
INTERACTION_DIR = PROJECT_DIR / "data" / "interaction"
INTERACTION_DIR.mkdir(parents=True, exist_ok=True)
INTERACTION_PATH = INTERACTION_DIR / "user_interactions.csv"
interactions.to_csv(INTERACTION_PATH, index=False)
print("Saved:", INTERACTION_PATH)

Saved: C:\Users\Ajay\Desktop\TED_Talks_Recommendation_Ajay\data\interaction\user_interactions.csv


## 4. Build the user-item interaction matrix

In [11]:
user_item_matrix = interactions.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating",
    aggfunc="mean",
)
print("User-item matrix shape:", user_item_matrix.shape)
display(user_item_matrix.iloc[:8, :10])

User-item matrix shape: (18, 100)


talk_id,Adam Savage: My obsession with objects and the stories they tell,"Aicha el-Wafi + Phyllis Rodriguez: The mothers who found forgiveness, friendship",Alex Steffen: The route to a sustainable future,Amber Case: We are all cyborgs now,Ameera Harouda: Why I put myself in danger to tell the stories of Gaza,Anders Fjellberg: Two nameless bodies washed up on the beach. Here are their stories,"Andrew Solomon: Love, no matter what",Andrew Stanton: The clues to a great story,Andrew Youn: 3 reasons why we can win the fight against poverty,Anna Mracek Dietrich: A plane you can drive
user_id,,,,,,,,,,
user_01,3.0,NaN,NaN,3.0,NaN,NaN,NaN,3.0,NaN,NaN
user_02,NaN,3.0,NaN,NaN,3.0,3.0,3.0,NaN,NaN,NaN
user_03,NaN,5.0,NaN,NaN,4.0,4.0,NaN,5.0,4.0,NaN
user_04,NaN,NaN,NaN,3.0,NaN,2.0,NaN,4.0,NaN,NaN
user_05,4.0,NaN,NaN,2.0,NaN,3.0,NaN,3.0,NaN,NaN
user_06,4.0,NaN,NaN,NaN,NaN,2.0,3.0,4.0,NaN,3.0
user_07,2.0,3.0,4.0,NaN,NaN,NaN,3.0,3.0,2.0,4.0
user_08,3.0,3.0,3.0,NaN,3.0,3.0,2.0,NaN,4.0,NaN


## 5. Basic interaction analysis

In [12]:
print("Unique users:", interactions["user_id"].nunique())
print("Unique TED Talks:", interactions["talk_id"].nunique())
print("Most active users:")
display(interactions.groupby("user_id").size().sort_values(ascending=False).head(10).to_frame("number_of_interactions"))

catalog_lookup = catalog.set_index("talk_id")["title"]
talk_stats = interactions.groupby("talk_id").agg(
    interaction_count=("rating", "size"),
    average_rating=("rating", "mean"),
    like_count=("liked", "sum"),
).sort_values(["interaction_count", "average_rating"], ascending=False)
talk_stats["title"] = talk_stats.index.map(catalog_lookup)
display(talk_stats.head(10))

Unique users: 18
Unique TED Talks: 100
Most active users:


,number_of_interactions
user_id,
user_01,45
user_02,45
user_03,45
user_04,45
user_05,45
user_06,45
user_07,45
user_08,45
user_09,45


,interaction_count,average_rating,like_count,title
talk_id,,,,
Robyn Stein DeLuca: The good news about PMS,15,3.266667,4,The good news about PMS
Siegfried Woldhek: The search for the true face of Leonardo,13,3.153846,4,The search for the true face of Leonardo
Andrew Stanton: The clues to a great story,12,3.666667,7,The clues to a great story
David Pogue: 10 top time-saving tech tips,12,3.083333,4,10 top time-saving tech tips
Sonia Shah: 3 reasons we still haven’t gotten rid of malaria,12,3.000000,3,3 reasons we still haven’t gotten rid of malaria
Tim Leberecht: 3 ways to (usefully) lose control of your brand,11,3.818182,5,3 ways to (usefully) lose control of your brand
Marcus Byrne: The dance of the dung beetle,11,3.545455,6,The dance of the dung beetle
Jamila Raqib: The secret to effective nonviolent resistance,11,3.454545,5,The secret to effective nonviolent resistance
Charles Robertson: Africa's next boom,11,3.272727,4,Africa's next boom


## 6. What is collaborative filtering?

Collaborative filtering uses patterns in user-item interactions. If two users have rated several talks similarly, the system may treat them as similar and use one user's ratings to help recommend talks to the other. This is different from content-based filtering, which compares the text or metadata of talks.

**Limitations of this exercise:** the interaction data is simulated; it has far fewer users than a real service; ratings are sparse; and the artificial interest groups make the patterns more structured than organic behaviour. Do not present these results as real user preference findings.